In [ ]:
# Create Spark session 
from pyspark.sql import SparkSession
spark=(
    SparkSession
    .builder
    .appName("Spark Pipelining")
    .master("local[*]")
    .config("spark.executor.cores",16)
    .config("spark.cores.max",4)
    .config("spark.executor.memory","512M")
    .getOrCreate()
)

In [10]:
spark

In [ ]:
# Get default parallelism (usually equals number of cores available)
.sparkContext.defaultParallelism

8

In [ ]:
# Disable AQE
.conf.set("spark.sql.adaptive.enabled", False)
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", False)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

In [ ]:
# Reading  employee_records 
_schema="first_name string, last_name string, job_title string, dob string, email string, phone string, salary double, department_id int"
emp=spark.read.format("csv").schema(_schema).option("header",True).load("employee_records.txt")

In [ ]:
# Calculate average salary per department
from pyspark.sql.functions import avg
emp_avg=emp.groupBy("department_id").agg(avg("salary").alias("avg_sal"))

In [27]:
emp_avg.write.format("noop").mode("overwrite").save()

In [ ]:
# Import function to check partition ID

spark.conf.get("spark.sql.shuffle.partitions")


'200'

In [29]:
from pyspark.sql.functions import spark_partition_id
emp.withColumn("partition_id",spark_partition_id()).where("partition_id=0").show()

+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+------------+
|first_name| last_name|           job_title|       dob|               email|               phone|  salary|department_id|partition_id|
+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+------------+
|   Richard|  Morrison|Public relations ...|1973-05-05|melissagarcia@exa...|       (699)525-4827|512653.0|            8|           0|
|     Bobby|  Mccarthy|   Barrister's clerk|1974-04-25|   llara@example.net|  (750)846-1602x7458|999836.0|            7|           0|
|    Dennis|    Norman|Land/geomatics su...|1990-06-24| jturner@example.net|    873.820.0518x825|131900.0|           10|           0|
|      John|    Monroe|        Retail buyer|1968-06-16|  erik33@example.net|    820-813-0557x624|485506.0|            1|           0|
|  Michelle|   Elliott|      Air cabin crew|1975-03-31|tiffany

In [23]:
_schema="first_name string, last_name string, job_title string, dob string, email string, phone string, salary double, department_id int"
emp_part=spark.read.format("csv").schema(_schema).option("header",True).load("employee_records.txt")

In [ ]:
# Perform same aggregation again on new DataFrame
emp_avg=emp_part.groupBy("department_id").agg(avg("salary").alias("avg_sal"))

In [ ]:
emp_avg.write.format("noop").mode("overwrite")